In [0]:
"""
01_parse_events.py

Parses Bronze manufacturing events into a structured
Silver streaming table.

Input:
    manufacturing_events

Output:
    parsed_events

Author:
Sumanth Vempalle

Version:
2.2.1
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import (
    col,
    current_timestamp,
    from_json,
)

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    TimestampType,
    StructField,
    StructType,
)

# ============================================================
# Payload Schema
# ============================================================

payload_schema = StructType([

    StructField("operation_number", IntegerType()),
    StructField("operation_name", StringType()),
    StructField("department", StringType()),

    StructField("machine_id", StringType()),
    StructField("line_id", StringType()),
    StructField("hall_id", StringType()),

    StructField("machine_name", StringType()),
    StructField("machine_type", StringType()),

    StructField("station_code", StringType()),
    StructField("station_type", StringType()),

    StructField("line_name", StringType()),
    StructField("hall_name", StringType()),

    StructField("tool_name", StringType()),
    StructField("tool_type", StringType()),

    StructField("operator_name", StringType()),
    StructField("skill_level", StringType()),

    StructField("target_force_kn", DoubleType()),
    StructField("actual_force_kn", DoubleType()),
    StructField("force_deviation_kn", DoubleType()),

    StructField("displacement_mm", DoubleType()),
    StructField("cycle_time_sec", IntegerType()),

    StructField("quality_result", StringType()),

    StructField("test_result_id", StringType()),
    StructField("test_program_id", StringType()),
    StructField("test_name", StringType()),
    StructField("target_value", DoubleType()),
    StructField("measured_value", DoubleType()),
    StructField("unit", StringType()),
    StructField("result", StringType()),

    StructField("scan_id", StringType()),
    StructField("material_number", StringType()),
    StructField("batch_number", StringType()),
    StructField("supplier", StringType()),
    StructField("scan_status", StringType()),

    StructField("package_id", StringType()),
    StructField("package_type", StringType()),
    StructField("package_weight_kg", DoubleType()),
    StructField("package_length_mm", IntegerType()),
    StructField("package_width_mm", IntegerType()),
    StructField("package_height_mm", IntegerType()),
    StructField("packaging_status", StringType()),

    StructField("sap_order_number", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("priority", StringType()),
    StructField("planned_shift", StringType()),
    StructField("routing_version", StringType()),
    StructField("planner", StringType()),
    StructField("status", StringType()),

    StructField("production_line", StringType()),

    StructField("product_name", StringType()),
    StructField("family", StringType()),
    StructField("rated_voltage_kv", DoubleType())

])

# ============================================================
# Event Schema
# ============================================================

event_schema = StructType([

    StructField("event_id", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("event_type", StringType()),
    StructField("event_version", StringType()),

    StructField("plant_code", StringType()),
    StructField("hall_id", StringType()),
    StructField("line_id", StringType()),
    StructField("machine_id", StringType()),

    StructField("execution_id", StringType()),
    StructField("work_order_id", StringType()),
    StructField("serial_number", StringType()),
    StructField("operator_id", StringType()),
    StructField("product_code", StringType()),

    StructField("source_system", StringType()),
    StructField("correlation_id", StringType()),

    StructField(
        "payload",
        payload_schema,
    ),

])

# ============================================================
# Bronze Wrapper Schema
# ============================================================

bronze_schema = StructType([

    StructField("kafka_topic", StringType()),
    StructField("kafka_partition", IntegerType()),
    StructField("kafka_offset", LongType()),
    StructField("ingestion_timestamp", TimestampType()),

    StructField(
        "event",
        event_schema,
    ),

])

# ============================================================
# Parsed Events
# ============================================================

@dp.table(
    name="parsed_events",
    comment="Parsed manufacturing events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_event_timestamp",
    "event_timestamp IS NOT NULL",
)
def parsed_events():

    bronze = spark.readStream.table(
        "manufacturing_events"
    )

    parsed = (

        bronze

        .withColumn(

            "parsed",

            from_json(

                col("raw_event"),

                bronze_schema,

            ),

        )

        .withColumn(

            "silver_processing_timestamp",

            current_timestamp()

        )

    )

    return (

        parsed.select(

            # ====================================================
            # Kafka Metadata
            # ====================================================

            col("parsed.kafka_topic"),

            col("parsed.kafka_partition"),

            col("parsed.kafka_offset"),

            col("parsed.ingestion_timestamp").alias(
                "bronze_ingestion_timestamp"
            ),

            # ====================================================
            # Audit
            # ====================================================

            col("silver_processing_timestamp"),

            # ====================================================
            # Manufacturing Event
            # ====================================================

            col("parsed.event.*"),

        )

    )